# 智慧垃圾分類 — YOLO26m 訓練 Notebook (v5)

**ADR-001 對齊：YOLO26m + TensorRT FP16 部署**

**執行前必做：Runtime → Change runtime type → T4 GPU**

## 執行順序
1. Cell 1：確認 GPU
2. Cell 2：安裝套件
3. Cell 3：Clone 專案
4. Cell 4：下載 Roboflow 資料集（需貼上 API Key）
5. Cell 5：類別對應（42 類 → 6 類）
6. Cell 6：訓練 yolo26m → waste_sorter_v5
7. Cell 7：下載 best_v5.pt

## 部署流程（訓練完成後）
```
① Colab 訓練 → ② Windows 匯出 ONNX → ③ SCP to Jetson
④ Jetson: trtexec → best_v5.engine
⑤ Jetson: python live_detect.py（pycuda + TRT）
```

## 六大類別（ADR-001）
| ID | 中文 | 英文 |
|----|------|------|
| 0 | 寶特瓶 | Plastic Bottle |
| 1 | 鐵鋁罐 | Metal Can |
| 2 | 紙餐盒 | Paper Box |
| 3 | 塑膠袋 | Plastic Bag |
| 4 | 鋁箔包 | Foil/Carton |
| 5 | 一般垃圾 | General Waste |

In [ ]:
# ── Cell 1：確認 GPU ───────────────────────────────────────────────────────────
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('❌ 沒有 GPU，請先到 Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2：安裝套件 ───────────────────────────────────────────────────────────
!pip install ultralytics roboflow pyyaml -q
print('套件安裝完成')

In [ ]:
# ── Cell 3：Clone 專案 ─────────────────────────────────────────────────────────
import os

REPO = 'AI-course'
if os.path.exists(REPO):
    print('Repo 已存在，執行 git pull...')
    !cd {REPO} && git pull
else:
    !git clone https://github.com/Saibusu/AI-course.git

%cd /content/AI-course
print('工作目錄：', os.getcwd())

In [ ]:
# ── Cell 4：下載 Roboflow 資料集 ───────────────────────────────────────────────
# 資料集：YOLO Waste Detection (ProjectVerba) — 42 類，真實 bounding box
# ⚠️ 勿將 API Key 上傳 GitHub，只在 Colab 執行時填入
import os
from roboflow import Roboflow

ROBOFLOW_API_KEY = 'YOUR_ROBOFLOW_API_KEY'  # ← 在這裡貼上你的 API Key，勿上傳 GitHub

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('projectverba').project('yolo-waste-detection')
version = project.version(1)
dataset = version.download('yolov8', location='data/roboflow_raw')

print('下載完成，路徑：', dataset.location)

# 確認 data.yaml
import glob
yaml_files = glob.glob('data/roboflow_raw/**/data.yaml', recursive=True)
print('data.yaml 位置：', yaml_files)

# 確認圖片數
for split in ['train', 'valid', 'test']:
    path = f'data/roboflow_raw/{split}/images'
    if os.path.exists(path):
        print(f'  {split}: {len(os.listdir(path))} images')

In [ ]:
# ── Cell 5：類別對應 42 → 6 ────────────────────────────────────────────────────
import yaml, glob, os, shutil, pathlib

TARGET_NAMES = ['寶特瓶', '鐵鋁罐', '紙餐盒', '塑膠袋', '鋁箔包', '一般垃圾']

# Roboflow 類別名稱 → 我們的 6 類別 ID
NAME_MAP = {
    # 寶特瓶 (0)
    'plastic bottle': 0, 'milk bottle': 0, 'plastic can': 0,
    'plastic canister': 0, 'plastic cup': 0,
    # 鐵鋁罐 (1)
    'aluminum can': 1, 'tin': 1, 'scrap metal': 1,
    'food can': 1, 'drink can': 1,
    # 紙餐盒 (2)
    'paper': 2, 'cardboard': 2, 'paper cups': 2, 'paper cup': 2,
    'paper bag': 2, 'disposable tableware': 2, 'papier mache': 2,
    'paper shavings': 2, 'cellulose': 2,
    # 塑膠袋 (3)
    'plastic bag': 3, 'zip plastic bag': 3, 'stretch film': 3,
    'combined plastic': 3, 'plastic film': 3,
    # 鋁箔包 (4)
    'foil': 4, 'tetra pack': 4, 'aluminium foil': 4,
    # 一般垃圾 (5) = 其餘所有類別（default）
}

# 讀取 data.yaml
yaml_path = glob.glob('data/roboflow_raw/**/data.yaml', recursive=True)[0]
with open(yaml_path) as f:
    orig_yaml = yaml.safe_load(f)

orig_names = orig_yaml.get('names', [])
if isinstance(orig_names, dict):
    orig_names = [orig_names[i] for i in sorted(orig_names.keys())]

print(f'原始類別數：{len(orig_names)}')

# 建立 ID 對應表
id_map = {}
for idx, name in enumerate(orig_names):
    target = NAME_MAP.get(name.strip().lower(), 5)
    id_map[idx] = target
    print(f'  {idx:2d}: {name:35s} → {target} ({TARGET_NAMES[target]})')

# 輸出目錄
out_dir = pathlib.Path('data/roboflow_6class')
dataset_dir = pathlib.Path(yaml_path).parent

counters = {i: 0 for i in range(6)}

for split in ['train', 'valid', 'test']:
    img_src = dataset_dir / split / 'images'
    lbl_src = dataset_dir / split / 'labels'
    if not img_src.exists():
        continue

    out_split = 'val' if split == 'valid' else split
    out_img = out_dir / out_split / 'images'
    out_lbl = out_dir / out_split / 'labels'
    out_img.mkdir(parents=True, exist_ok=True)
    out_lbl.mkdir(parents=True, exist_ok=True)

    for img_path in img_src.glob('*.[jJpP][pPnN][gG]'):
        shutil.copy2(img_path, out_img / img_path.name)
        lbl_path = lbl_src / (img_path.stem + '.txt')
        out_lbl_path = out_lbl / (img_path.stem + '.txt')

        if lbl_path.exists():
            new_lines = []
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if not parts:
                        continue
                    new_cls = id_map.get(int(parts[0]), 5)
                    counters[new_cls] += 1
                    new_lines.append(f'{new_cls} ' + ' '.join(parts[1:]))
            with open(out_lbl_path, 'w') as f:
                f.write('\n'.join(new_lines) + ('\n' if new_lines else ''))
        else:
            out_lbl_path.touch()

# 寫新 data.yaml
new_yaml_content = f"""path: {out_dir.resolve()}
train: train/images
val:   val/images
test:  test/images

nc: 6
names:
  0: 寶特瓶
  1: 鐵鋁罐
  2: 紙餐盒
  3: 塑膠袋
  4: 鋁箔包
  5: 一般垃圾
"""
(out_dir / 'data.yaml').write_text(new_yaml_content, encoding='utf-8')

print('\n✅ 類別對應完成')
print(f'資料集位置：{out_dir}')
print('各類別 bbox 數量：')
for cid, cnt in counters.items():
    print(f'  {cid} {TARGET_NAMES[cid]}: {cnt}')

In [ ]:
# ── Cell 6：訓練 YOLO26m → waste_sorter_v5 ────────────────────────────────────
# ADR-001：yolo26m（detection），TRT FP16 部署目標
from ultralytics import YOLO
import os

DATA_YAML = 'data/roboflow_6class/data.yaml'
assert os.path.exists(DATA_YAML), f'找不到 {DATA_YAML}，請先執行 Cell 5'

model = YOLO('yolo26m.pt')   # YOLO26 detection variant
print(f'模型：{model.info()}')

results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=8,
    device=0,
    project='runs/train',
    name='waste_sorter_v5',
    exist_ok=True,
    patience=20,
    cos_lr=True,
    lr0=1e-3,
    lrf=1e-2,
    mosaic=1.0,
    fliplr=0.5,
    degrees=15.0,
    translate=0.1,
    scale=0.5,
    hsv_s=0.7,
    hsv_v=0.4,
    mixup=0.1,
)

mAP = results.results_dict.get('metrics/mAP50(B)', 'N/A')
print(f'\n訓練完成！mAP@50 = {mAP}')
print('模型路徑：runs/train/waste_sorter_v5/weights/best.pt')

In [ ]:
# ── Cell 7：下載 best_v5.pt ───────────────────────────────────────────────────
import glob, shutil, os
from google.colab import files

pts = glob.glob('**/best.pt', recursive=True)
print('找到的 best.pt：', pts)

src = next((p for p in pts if 'waste_sorter_v5' in p), None)
if src is None:
    src = next(p for p in pts if p != 'best_v5.pt')

print(f'使用模型：{src}')
shutil.copy(src, 'best_v5.pt')
print(f'Model size: {os.path.getsize("best_v5.pt")/1e6:.1f} MB')

files.download('best_v5.pt')
print('\n✅ 下載完成')
print('接著在筆電執行：')
print('python -c "from ultralytics import YOLO; YOLO(\'best_v5.pt\').export(format=\'onnx\', imgsz=416, simplify=True, opset=12)"')
print('scp best_v5.onnx jetson@172.20.10.2:~/AI-course/models/best_v5.onnx')
print('然後在 Jetson 執行：')
print('/usr/src/tensorrt/bin/trtexec --onnx=models/best_v5.onnx --saveEngine=models/best_v5.engine --fp16')